## Experiment Set 2: Dynamic Batching, max batch size 2

Dynamic batching, poisson arrivals, asynchronous calls, varying arrival rates

To edit the model configuration:

``` bash
# runs on node-serve-system
nano ~/serve-system-chi/models/food_classifier_onnx/config.pbtxt
```

Current config:

``` bash
name: "food_classifier_onnx"
backend: "onnxruntime"
max_batch_size: 2
input [
  {
    name: "input"  # has to match ONNX model's input name
    data_type: TYPE_FP32
    dims: [3, 224, 224]  # has to match ONNX input shape
  }
]
output [
  {
    name: "output"  # has to match ONNX model output name
    data_type: TYPE_FP32  # output is a list of probabilities
    dims: [11]  #
  }
]
  instance_group [
    {
      count: 1
      kind: KIND_GPU
      gpus: [ 0 ]
    }
]

dynamic_batching {
    preferred_batch_size: [2]
    max_queue_delay_microseconds: 100
}

```
Save the file (use Ctrl+O then Enter, then Ctrl+X).

Re-build the container image with this change:

``` bash
# runs on node-serve-system
docker compose -f ~/serve-system-chi/docker/docker-compose-triton.yaml build triton_server
```

and then bring the server back up:

``` bash
# runs on node-serve-system
docker compose -f ~/serve-system-chi/docker/docker-compose-triton.yaml up triton_server --force-recreate -d
```

and use

``` bash
# runs on node-serve-system
docker logs triton_server
```

to make sure the server comes up and is ready.

Before we benchmark this service again, let’s get some pre-benchmark stats about how many requests have been served, broken down by batch size. (If you’ve just restarted the server, it would be zero!)

In [7]:
!curl -s http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats | python -m json.tool > /home/jovyan/work/batch_2_results/stats_before.json

In [8]:
# runs inside the Jupyter container on node-serve-system
perf_analyzer -u triton_server:8000 -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 --request-rate-range 80 --async --request-distribution poisson

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average throughput
  Measurement window: 5000 msec
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 80 inference requests per second
  Client: 
    Request count: 1449
    Throughput: 78.2926 infer/sec
    Avg latency: 9389 usec (standard deviation 3561 usec)
    p50 latency: 8236 usec
    p90 latency: 13179 usec
    p95 latency: 14676 usec
    p99 latency: 17450 usec
    Avg HTTP time: 9255 usec (send/recv 414 usec + response wait 8841 usec)
  Server: 
    Inference count: 1449
    Execution count: 1381
    Successful request count: 1449
    Avg request latency: 7025 usec (overhead 36 usec + queue 1568 usec + compute input 163 usec + compute infer 5237 usec + compute output 20 usec)
Inferences/Second vs. Client Average Batch Latency
Request Rate: 80, throughput: 78.2926 infer/sec, latency 9389 us

In [11]:
curl -s http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats | python -m json.tool > /home/jovyan/work/batch_2_results/stats_80.json

In [12]:
# runs inside the Jupyter container on node-serve-system
perf_analyzer -u triton_server:8000 -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 --request-rate-range 100 --async --request-distribution poisson

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average throughput
  Measurement window: 5000 msec
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 100 inference requests per second
  Client: 
    Request count: 1795
    Throughput: 97.8442 infer/sec
    Avg latency: 9985 usec (standard deviation 3099 usec)
    p50 latency: 8753 usec
    p90 latency: 14482 usec
    p95 latency: 16208 usec
    p99 latency: 19462 usec
    Avg HTTP time: 9945 usec (send/recv 571 usec + response wait 9374 usec)
  Server: 
    Inference count: 1795
    Execution count: 1649
    Successful request count: 1795
    Avg request latency: 7287 usec (overhead 37 usec + queue 1950 usec + compute input 174 usec + compute infer 5106 usec + compute output 19 usec)
Inferences/Second vs. Client Average Batch Latency
Request Rate: 100, throughput: 97.8442 infer/sec, latency 9985 

In [13]:
curl -s http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats | python -m json.tool > /home/jovyan/work/batch_2_results/stats_100.json

In [14]:
# runs inside the Jupyter container on node-serve-system
perf_analyzer -u triton_server:8000 -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 --request-rate-range 120 --async --request-distribution poisson

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average throughput
  Measurement window: 5000 msec
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 120 inference requests per second
  Client: 
    Request count: 2249
    Throughput: 121.015 infer/sec
    Avg latency: 9206 usec (standard deviation 3557 usec)
    p50 latency: 8236 usec
    p90 latency: 13271 usec
    p95 latency: 15014 usec
    p99 latency: 18458 usec
    Avg HTTP time: 9103 usec (send/recv 601 usec + response wait 8502 usec)
  Server: 
    Inference count: 2249
    Execution count: 2052
    Successful request count: 2249
    Avg request latency: 6716 usec (overhead 33 usec + queue 1942 usec + compute input 174 usec + compute infer 4549 usec + compute output 17 usec)
Inferences/Second vs. Client Average Batch Latency
Request Rate: 120, throughput: 121.015 infer/sec, latency 9206 

In [15]:
curl -s http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats | python -m json.tool > /home/jovyan/work/batch_2_results/stats_120.json

In [16]:
# runs inside the Jupyter container on node-serve-system
perf_analyzer -u triton_server:8000 -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 --request-rate-range 140 --async --request-distribution poisson

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average throughput
  Measurement window: 5000 msec
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 140 inference requests per second
  Client: 
    Request count: 2530
    Throughput: 136.668 infer/sec
    Avg latency: 9481 usec (standard deviation 3080 usec)
    p50 latency: 8632 usec
    p90 latency: 13578 usec
    p95 latency: 15160 usec
    p99 latency: 19602 usec
    Avg HTTP time: 9450 usec (send/recv 695 usec + response wait 8755 usec)
  Server: 
    Inference count: 2530
    Execution count: 2262
    Successful request count: 2530
    Avg request latency: 6660 usec (overhead 31 usec + queue 2069 usec + compute input 162 usec + compute infer 4380 usec + compute output 16 usec)
Inferences/Second vs. Client Average Batch Latency
Request Rate: 140, throughput: 136.668 infer/sec, latency 9481 

In [17]:
curl -s http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats | python -m json.tool > /home/jovyan/work/batch_2_results/stats_140.json

In [18]:
# runs inside the Jupyter container on node-serve-system
perf_analyzer -u triton_server:8000 -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 --request-rate-range 160 --async --request-distribution poisson

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average throughput
  Measurement window: 5000 msec
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 160 inference requests per second
  Client: 
    Request count: 2890
    Throughput: 155.385 infer/sec
    Avg latency: 9123 usec (standard deviation 3911 usec)
    p50 latency: 8249 usec
    p90 latency: 13077 usec
    p95 latency: 14450 usec
    p99 latency: 18479 usec
    Avg HTTP time: 9021 usec (send/recv 646 usec + response wait 8375 usec)
  Server: 
    Inference count: 2890
    Execution count: 2557
    Successful request count: 2890
    Avg request latency: 6286 usec (overhead 29 usec + queue 2040 usec + compute input 158 usec + compute infer 4043 usec + compute output 15 usec)
Inferences/Second vs. Client Average Batch Latency
Request Rate: 160, throughput: 155.385 infer/sec, latency 9123 

In [19]:
curl -s http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats | python -m json.tool > /home/jovyan/work/batch_2_results/stats_160.json

In [20]:
# runs inside the Jupyter container on node-serve-system
perf_analyzer -u triton_server:8000 -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 --request-rate-range 180 --async --request-distribution poisson

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average throughput
  Measurement window: 5000 msec
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 180 inference requests per second
  Client: 
    Request count: 3419
    Throughput: 182.254 infer/sec
    Avg latency: 8152 usec (standard deviation 3273 usec)
    p50 latency: 7372 usec
    p90 latency: 11742 usec
    p95 latency: 12943 usec
    p99 latency: 16681 usec
    Avg HTTP time: 8067 usec (send/recv 407 usec + response wait 7660 usec)
  Server: 
    Inference count: 3419
    Execution count: 3038
    Successful request count: 3419
    Avg request latency: 5581 usec (overhead 25 usec + queue 1835 usec + compute input 143 usec + compute infer 3563 usec + compute output 13 usec)
Inferences/Second vs. Client Average Batch Latency
Request Rate: 180, throughput: 182.254 infer/sec, latency 8152 

In [21]:
curl -s http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats | python -m json.tool > /home/jovyan/work/batch_2_results/stats_180.json

In [22]:
# runs inside the Jupyter container on node-serve-system
perf_analyzer -u triton_server:8000 -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 --request-rate-range 200 --async --request-distribution poisson

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average throughput
  Measurement window: 5000 msec
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 200 inference requests per second
  Client: 
    Request count: 3702
    Throughput: 197.206 infer/sec
    Avg latency: 8740 usec (standard deviation 3736 usec)
    p50 latency: 7930 usec
    p90 latency: 12400 usec
    p95 latency: 13862 usec
    p99 latency: 19747 usec
    Avg HTTP time: 8684 usec (send/recv 695 usec + response wait 7989 usec)
  Server: 
    Inference count: 3702
    Execution count: 3199
    Successful request count: 3702
    Avg request latency: 5657 usec (overhead 26 usec + queue 1996 usec + compute input 151 usec + compute infer 3470 usec + compute output 13 usec)
Inferences/Second vs. Client Average Batch Latency
Request Rate: 200, throughput: 197.206 infer/sec, latency 8740 

In [23]:
curl -s http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats | python -m json.tool > /home/jovyan/work/batch_2_results/stats_200.json

In [24]:
# runs inside the Jupyter container on node-serve-system
perf_analyzer -u triton_server:8000 -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 --request-rate-range 220 --async --request-distribution poisson

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average throughput
  Measurement window: 5000 msec
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 220 inference requests per second
  Client: 
    Request count: 4193
    Avg send request rate: 232.02 infer/sec
    [WARNING] Perf Analyzer was not able to keep up with the desired request rate. 1.67% of the requests were delayed. 
    Throughput: 216.70 infer/sec
    Avg latency: 8846 usec (standard deviation 5760 usec)
    p50 latency: 7733 usec
    p90 latency: 12289 usec
    p95 latency: 14407 usec
    p99 latency: 26950 usec
    Avg HTTP time: 8684 usec (send/recv 822 usec + response wait 7862 usec)
  Server: 
    Inference count: 4193
    Execution count: 3608
    Successful request count: 4193
    Avg request latency: 5349 usec (overhead 24 usec + queue 1929 usec + compute input 144 usec + c

In [25]:
curl -s http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats | python -m json.tool > /home/jovyan/work/batch_2_results/stats_220.json

In [26]:
# runs inside the Jupyter container on node-serve-system
perf_analyzer -u triton_server:8000 -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 --request-rate-range 240 --async --request-distribution poisson > /home/jovyan/work/batch_2_results/batch_2_240.txt

In [27]:
curl -s http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats | python -m json.tool > /home/jovyan/work/batch_2_results/stats_240.json

In [28]:
# runs inside the Jupyter container on node-serve-system
perf_analyzer -u triton_server:8000 -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 --request-rate-range 260 --async --request-distribution poisson | tee /home/jovyan/work/batch_2_results/batch_2_260.txt

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average throughput
  Measurement window: 5000 msec
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 260 inference requests per second
  Client: 
    Request count: 4955
    Avg send request rate: 268.91 infer/sec
    [WARNING] Perf Analyzer was not able to keep up with the desired request rate. 1.41% of the requests were delayed. 
    Throughput: 258.57 infer/sec
    Avg latency: 8893 usec (standard deviation 5798 usec)
    p50 latency: 7833 usec
    p90 latency: 12179 usec
    p95 latency: 14590 usec
    p99 latency: 29891 usec
    Avg HTTP time: 8753 usec (send/recv 633 usec + response wait 8120 usec)
  Server: 
    Inference count: 4955
    Execution count: 4108
    Successful request count: 4955
    Avg request latency: 5168 usec (overhead 22 usec + queue 2006 usec + compute input 148 usec + c

In [29]:
curl -s http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats | python -m json.tool | tee /home/jovyan/work/batch_2_results/stats_260.json

{
    "model_stats": [
        {
            "name": "food_classifier_onnx",
            "version": "1",
            "last_inference": 1778447036903,
            "inference_count": 48870,
            "execution_count": 42308,
            "inference_stats": {
                "success": {
                    "count": 48870,
                    "ns": 2856687512296
                },
                "fail": {
                    "count": 0,
                    "ns": 0
                },
                "queue": {
                    "count": 48870,
                    "ns": 2660264906500
                },
                "compute_input": {
                    "count": 48870,
                    "ns": 7815844339
                },
                "compute_infer": {
                    "count": 48870,
                    "ns": 186620699416
                },
                "compute_output": {
                    "count": 48870,
                    "ns": 694796765
                },
       

In [30]:
# runs inside the Jupyter container on node-serve-system
perf_analyzer -u triton_server:8000 -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 --request-rate-range 280 --async --request-distribution poisson | tee /home/jovyan/work/batch_2_results/batch_2_280.txt

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average throughput
  Measurement window: 5000 msec
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 280 inference requests per second
  Client: 
    Request count: 5748
    Avg send request rate: 289.60 infer/sec
    [WARNING] Perf Analyzer was not able to keep up with the desired request rate. 2.26% of the requests were delayed. 
    Throughput: 288.98 infer/sec
    Avg latency: 8804 usec (standard deviation 5873 usec)
    p50 latency: 7667 usec
    p90 latency: 12349 usec
    p95 latency: 15415 usec
    p99 latency: 29570 usec
    Avg HTTP time: 8688 usec (send/recv 558 usec + response wait 8130 usec)
  Server: 
    Inference count: 5749
    Execution count: 4684
    Successful request count: 5749
    Avg request latency: 5171 usec (overhead 22 usec + queue 2105 usec + compute input 164 usec + c

In [31]:
curl -s http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats | python -m json.tool > /home/jovyan/work/batch_2_results/stats_280.json

In [32]:
# runs inside the Jupyter container on node-serve-system
perf_analyzer -u triton_server:8000 -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 --request-rate-range 300 --async --request-distribution poisson | tee /home/jovyan/work/batch_2_results/batch_2_300.txt

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average throughput
  Measurement window: 5000 msec
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 300 inference requests per second
  Client: 
    Request count: 6070
    Avg send request rate: 294.38 infer/sec
    [WARNING] Perf Analyzer was not able to keep up with the desired request rate. 2.67% of the requests were delayed. 
    Throughput: 299.95 infer/sec
    Avg latency: 9483 usec (standard deviation 8663 usec)
    p50 latency: 7958 usec
    p90 latency: 13444 usec
    p95 latency: 17948 usec
    p99 latency: 37577 usec
    Avg HTTP time: 9267 usec (send/recv 490 usec + response wait 8777 usec)
  Server: 
    Inference count: 6071
    Execution count: 4797
    Successful request count: 6071
    Avg request latency: 5451 usec (overhead 22 usec + queue 2366 usec + compute input 172 usec + c

In [33]:
curl -s http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats | python -m json.tool | tee /home/jovyan/work/batch_2_results/stats_300.json

{
    "model_stats": [
        {
            "name": "food_classifier_onnx",
            "version": "1",
            "last_inference": 1778447218785,
            "inference_count": 83111,
            "execution_count": 69846,
            "inference_stats": {
                "success": {
                    "count": 83111,
                    "ns": 3043503753049
                },
                "fail": {
                    "count": 0,
                    "ns": 0
                },
                "queue": {
                    "count": 83111,
                    "ns": 2741399556619
                },
                "compute_input": {
                    "count": 83111,
                    "ns": 13491304660
                },
                "compute_infer": {
                    "count": 83111,
                    "ns": 285495560294
                },
                "compute_output": {
                    "count": 83111,
                    "ns": 1083613318
                },
     

In [34]:
# runs inside the Jupyter container on node-serve-system
perf_analyzer -u triton_server:8000 -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 --request-rate-range 320 --async --request-distribution poisson | tee /home/jovyan/work/batch_2_results/batch_2_320.txt

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average throughput
  Measurement window: 5000 msec
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 320 inference requests per second
  Client: 
    Request count: 7048
    Avg send request rate: 324.03 infer/sec
    [WARNING] Perf Analyzer was not able to keep up with the desired request rate. 4.33% of the requests were delayed. 
    Throughput: 328.13 infer/sec
    Avg latency: 11156 usec (standard deviation 10975 usec)
    p50 latency: 8730 usec
    p90 latency: 16553 usec
    p95 latency: 27600 usec
    p99 latency: 53272 usec
    Avg HTTP time: 10929 usec (send/recv 754 usec + response wait 10175 usec)
  Server: 
    Inference count: 7048
    Execution count: 5331
    Successful request count: 7048
    Avg request latency: 6566 usec (overhead 23 usec + queue 3346 usec + compute input 180 usec

In [35]:
curl -s http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats | python -m json.tool | tee /home/jovyan/work/batch_2_results/stats_320.json

{
    "model_stats": [
        {
            "name": "food_classifier_onnx",
            "version": "1",
            "last_inference": 1778447269946,
            "inference_count": 94597,
            "execution_count": 78612,
            "inference_stats": {
                "success": {
                    "count": 94597,
                    "ns": 3118243379605
                },
                "fail": {
                    "count": 0,
                    "ns": 0
                },
                "queue": {
                    "count": 94597,
                    "ns": 2779463690434
                },
                "compute_input": {
                    "count": 94597,
                    "ns": 15545361439
                },
                "compute_infer": {
                    "count": 94597,
                    "ns": 319720277668
                },
                "compute_output": {
                    "count": 94597,
                    "ns": 1219925551
                },
     

In [36]:
# runs inside the Jupyter container on node-serve-system
perf_analyzer -u triton_server:8000 -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 --request-rate-range 340 --async --request-distribution poisson | tee /home/jovyan/work/batch_2_results/batch_2_340.txt

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average throughput
  Measurement window: 5000 msec
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 340 inference requests per second
  Client: 
    Request count: 7138
    Avg send request rate: 337.98 infer/sec
    [WARNING] Perf Analyzer was not able to keep up with the desired request rate. 3.63% of the requests were delayed. 
    Throughput: 338.88 infer/sec
    Avg latency: 10354 usec (standard deviation 10480 usec)
    p50 latency: 8328 usec
    p90 latency: 14266 usec
    p95 latency: 22028 usec
    p99 latency: 51319 usec
    Avg HTTP time: 10147 usec (send/recv 514 usec + response wait 9633 usec)
  Server: 
    Inference count: 7136
    Execution count: 5382
    Successful request count: 7136
    Avg request latency: 6130 usec (overhead 23 usec + queue 2988 usec + compute input 178 usec 

In [37]:
curl -s http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats | python -m json.tool | tee /home/jovyan/work/batch_2_results/stats_340.json

{
    "model_stats": [
        {
            "name": "food_classifier_onnx",
            "version": "1",
            "last_inference": 1778447351585,
            "inference_count": 119623,
            "execution_count": 96457,
            "inference_stats": {
                "success": {
                    "count": 119623,
                    "ns": 3325689159166
                },
                "fail": {
                    "count": 0,
                    "ns": 0
                },
                "queue": {
                    "count": 119623,
                    "ns": 2900230411827
                },
                "compute_input": {
                    "count": 119623,
                    "ns": 20457412922
                },
                "compute_infer": {
                    "count": 119623,
                    "ns": 400581271656
                },
                "compute_output": {
                    "count": 119623,
                    "ns": 1528058340
                },

In [38]:
# runs inside the Jupyter container on node-serve-system
perf_analyzer -u triton_server:8000 -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 --request-rate-range 360 --async --request-distribution poisson | tee /home/jovyan/work/batch_2_results/batch_2_360.txt

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average throughput
  Measurement window: 5000 msec
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 360 inference requests per second
  Client: 
    Request count: 7102
    Avg send request rate: 365.72 infer/sec
    [WARNING] Perf Analyzer was not able to keep up with the desired request rate. 2.34% of the requests were delayed. 
    Throughput: 356.90 infer/sec
    Avg latency: 10352 usec (standard deviation 8237 usec)
    p50 latency: 8721 usec
    p90 latency: 15111 usec
    p95 latency: 19431 usec
    p99 latency: 40611 usec
    Avg HTTP time: 10179 usec (send/recv 467 usec + response wait 9712 usec)
  Server: 
    Inference count: 7102
    Execution count: 5242
    Successful request count: 7102
    Avg request latency: 6170 usec (overhead 22 usec + queue 3051 usec + compute input 186 usec +

In [39]:
curl -s http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats | python -m json.tool | tee /home/jovyan/work/batch_2_results/stats_360.json

{
    "model_stats": [
        {
            "name": "food_classifier_onnx",
            "version": "1",
            "last_inference": 1778447435509,
            "inference_count": 144912,
            "execution_count": 114973,
            "inference_stats": {
                "success": {
                    "count": 144912,
                    "ns": 3494241172637
                },
                "fail": {
                    "count": 0,
                    "ns": 0
                },
                "queue": {
                    "count": 144912,
                    "ns": 2989833612003
                },
                "compute_input": {
                    "count": 144912,
                    "ns": 25185038779
                },
                "compute_infer": {
                    "count": 144912,
                    "ns": 473926279724
                },
                "compute_output": {
                    "count": 144912,
                    "ns": 1828620356
                }

In [40]:
# runs inside the Jupyter container on node-serve-system
perf_analyzer -u triton_server:8000 -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 --request-rate-range 380 --async --request-distribution poisson | tee /home/jovyan/work/batch_2_results/batch_2_380.txt

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average throughput
  Measurement window: 5000 msec
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 380 inference requests per second
  Client: 
    Request count: 7617
    Avg send request rate: 380.61 infer/sec
    [WARNING] Perf Analyzer was not able to keep up with the desired request rate. 2.64% of the requests were delayed. 
    Throughput: 377.80 infer/sec
    Avg latency: 10790 usec (standard deviation 9931 usec)
    p50 latency: 8888 usec
    p90 latency: 15476 usec
    p95 latency: 21231 usec
    p99 latency: 46985 usec
    Avg HTTP time: 10518 usec (send/recv 474 usec + response wait 10044 usec)
  Server: 
    Inference count: 7618
    Execution count: 5454
    Successful request count: 7618
    Avg request latency: 6501 usec (overhead 23 usec + queue 3391 usec + compute input 198 usec 

In [41]:
curl -s http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats | python -m json.tool | tee /home/jovyan/work/batch_2_results/stats_380.json

{
    "model_stats": [
        {
            "name": "food_classifier_onnx",
            "version": "1",
            "last_inference": 1778447537209,
            "inference_count": 171903,
            "execution_count": 134169,
            "inference_stats": {
                "success": {
                    "count": 171903,
                    "ns": 3683689239355
                },
                "fail": {
                    "count": 0,
                    "ns": 0
                },
                "queue": {
                    "count": 171903,
                    "ns": 3095351001266
                },
                "compute_input": {
                    "count": 171903,
                    "ns": 30564671117
                },
                "compute_infer": {
                    "count": 171903,
                    "ns": 551514738285
                },
                "compute_output": {
                    "count": 171903,
                    "ns": 2166259992
                }

In [42]:
# runs inside the Jupyter container on node-serve-system
perf_analyzer -u triton_server:8000 -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 --request-rate-range 400 --async --request-distribution poisson | tee /home/jovyan/work/batch_2_results/batch_2_400.txt

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average throughput
  Measurement window: 5000 msec
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 400 inference requests per second
  Client: 
    Request count: 7721
    Avg send request rate: 403.65 infer/sec
    [WARNING] Perf Analyzer was not able to keep up with the desired request rate. 1.71% of the requests were delayed. 
    Throughput: 400.02 infer/sec
    Avg latency: 10147 usec (standard deviation 5655 usec)
    p50 latency: 8998 usec
    p90 latency: 15634 usec
    p95 latency: 18819 usec
    p99 latency: 28011 usec
    Avg HTTP time: 10024 usec (send/recv 401 usec + response wait 9623 usec)
  Server: 
    Inference count: 7722
    Execution count: 5401
    Successful request count: 7722
    Avg request latency: 6442 usec (overhead 23 usec + queue 3330 usec + compute input 201 usec +

In [43]:
curl -s http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats | python -m json.tool | tee /home/jovyan/work/batch_2_results/stats_400.json

{
    "model_stats": [
        {
            "name": "food_classifier_onnx",
            "version": "1",
            "last_inference": 1778447626922,
            "inference_count": 197296,
            "execution_count": 151725,
            "inference_stats": {
                "success": {
                    "count": 197296,
                    "ns": 3870794997507
                },
                "fail": {
                    "count": 0,
                    "ns": 0
                },
                "queue": {
                    "count": 197296,
                    "ns": 3203290807188
                },
                "compute_input": {
                    "count": 197296,
                    "ns": 35711586950
                },
                "compute_infer": {
                    "count": 197296,
                    "ns": 624620351735
                },
                "compute_output": {
                    "count": 197296,
                    "ns": 2485440937
                }

In [44]:
# runs inside the Jupyter container on node-serve-system
perf_analyzer -u triton_server:8000 -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 --request-rate-range 420:500:20 --async --request-distribution poisson | tee /home/jovyan/work/batch_2_results/batch_2_420-500.txt

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average throughput
  Measurement window: 5000 msec
  Latency limit: 0 msec
  Request Rate limit: 500 requests per seconds
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 420 inference requests per second
  Client: 
    Request count: 8835
    Avg send request rate: 402.43 infer/sec
    [WARNING] Perf Analyzer was not able to keep up with the desired request rate. 3.89% of the requests were delayed. 
    Throughput: 416.63 infer/sec
    Avg latency: 11833 usec (standard deviation 12033 usec)
    p50 latency: 9206 usec
    p90 latency: 17615 usec
    p95 latency: 28945 usec
    p99 latency: 55144 usec
    Avg HTTP time: 11567 usec (send/recv 584 usec + response wait 10983 usec)
  Server: 
    Inference count: 8835
    Execution count: 6029
    Successful request count: 8835
    Avg request latency:

In [45]:
curl -s http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats | python -m json.tool | tee /home/jovyan/work/batch_2_results/stats_420-500.json

{
    "model_stats": [
        {
            "name": "food_classifier_onnx",
            "version": "1",
            "last_inference": 1778448693671,
            "inference_count": 302583,
            "execution_count": 219245,
            "inference_stats": {
                "success": {
                    "count": 302583,
                    "ns": 4825749216626
                },
                "fail": {
                    "count": 0,
                    "ns": 0
                },
                "queue": {
                    "count": 302583,
                    "ns": 3831716107371
                },
                "compute_input": {
                    "count": 302583,
                    "ns": 58738353544
                },
                "compute_infer": {
                    "count": 302583,
                    "ns": 924214548699
                },
                "compute_output": {
                    "count": 302583,
                    "ns": 3824000714
                }

In [46]:
# runs inside the Jupyter container on node-serve-system
perf_analyzer -u triton_server:8000 -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 --request-rate-range 520:600:20 --async --request-distribution poisson | tee /home/jovyan/work/batch_2_results/batch_2_520-600.txt

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average throughput
  Measurement window: 5000 msec
  Latency limit: 0 msec
  Request Rate limit: 600 requests per seconds
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 520 inference requests per second
  Client: 
    Request count: 11937
    Avg send request rate: 527.42 infer/sec
    [WARNING] Perf Analyzer was not able to keep up with the desired request rate. 5.43% of the requests were delayed. 
    Throughput: 527.35 infer/sec
    Avg latency: 19686 usec (standard deviation 24008 usec)
    p50 latency: 13045 usec
    p90 latency: 44593 usec
    p95 latency: 68865 usec
    p99 latency: 91343 usec
    Avg HTTP time: 19311 usec (send/recv 762 usec + response wait 18549 usec)
  Server: 
    Inference count: 11936
    Execution count: 6992
    Successful request count: 11936
    Avg request late

In [47]:
curl -s http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats | python -m json.tool | tee /home/jovyan/work/batch_2_results/stats_520-600.json

{
    "model_stats": [
        {
            "name": "food_classifier_onnx",
            "version": "1",
            "last_inference": 1778449440549,
            "inference_count": 507245,
            "execution_count": 333183,
            "inference_stats": {
                "success": {
                    "count": 507245,
                    "ns": 9096573410416
                },
                "fail": {
                    "count": 0,
                    "ns": 0
                },
                "queue": {
                    "count": 507245,
                    "ns": 7472238607480
                },
                "compute_input": {
                    "count": 507245,
                    "ns": 109276334901
                },
                "compute_infer": {
                    "count": 507245,
                    "ns": 1496020689483
                },
                "compute_output": {
                    "count": 507245,
                    "ns": 6479124549
               

In [48]:
# runs inside the Jupyter container on node-serve-system
perf_analyzer -u triton_server:8000 -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 --request-rate-range 620:800:20 --async --request-distribution poisson | tee /home/jovyan/work/batch_2_results/batch_2_620-800.txt

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average throughput
  Measurement window: 5000 msec
  Latency limit: 0 msec
  Request Rate limit: 800 requests per seconds
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 620 inference requests per second
  Client: 
    Request count: 13881
    Avg send request rate: 632.75 infer/sec
    [WARNING] Perf Analyzer was not able to keep up with the desired request rate. 4.87% of the requests were delayed. 
    Throughput: 632.93 infer/sec
    Avg latency: 100742 usec (standard deviation 28661 usec)
    p50 latency: 107879 usec
    p90 latency: 168393 usec
    p95 latency: 177734 usec
    p99 latency: 199273 usec
    Avg HTTP time: 100448 usec (send/recv 1303 usec + response wait 99145 usec)
  Server: 
    Inference count: 13882
    Execution count: 7056
    Successful request count: 13882
    Avg reque

In [49]:
curl -s http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats | python -m json.tool | tee /home/jovyan/work/batch_2_results/stats_620-660.json

{
    "model_stats": [
        {
            "name": "food_classifier_onnx",
            "version": "1",
            "last_inference": 1778450289015,
            "inference_count": 715010,
            "execution_count": 446201,
            "inference_stats": {
                "success": {
                    "count": 715010,
                    "ns": 103122372652446
                },
                "fail": {
                    "count": 0,
                    "ns": 0
                },
                "queue": {
                    "count": 715010,
                    "ns": 100850402218885
                },
                "compute_input": {
                    "count": 715010,
                    "ns": 164797590112
                },
                "compute_infer": {
                    "count": 715010,
                    "ns": 2079805443152
                },
                "compute_output": {
                    "count": 715010,
                    "ns": 9236102301
           

In [50]:
# runs inside the Jupyter container on node-serve-system
perf_analyzer -u triton_server:8000 -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 --request-rate-range 20:60:20 --async --request-distribution poisson | tee /home/jovyan/work/batch_2_results/batch_2_20-60.txt

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average throughput
  Measurement window: 5000 msec
  Latency limit: 0 msec
  Request Rate limit: 60 requests per seconds
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 20 inference requests per second
  Client: 
    Request count: 335
    Throughput: 18.4946 infer/sec
    Avg latency: 8727 usec (standard deviation 1360 usec)
    p50 latency: 8359 usec
    p90 latency: 9655 usec
    p95 latency: 11409 usec
    p99 latency: 14604 usec
    Avg HTTP time: 8709 usec (send/recv 868 usec + response wait 7841 usec)
  Server: 
    Inference count: 335
    Execution count: 331
    Successful request count: 335
    Avg request latency: 6166 usec (overhead 38 usec + queue 536 usec + compute input 169 usec + compute infer 5400 usec + compute output 22 usec)
Request Rate: 40 inference requests per second
  Cl